# Cost and Latency Crossover Analysis
## Serverless vs Container vs Managed Endpoint ML Inference

This notebook runs the complete analysis pipeline:
1. Generate simulated measurement data
2. Run the cost model simulation
3. Compute crossover points
4. Perform sensitivity analysis
5. Generate all publication figures

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from cost_model.pricing import PricingConfig, get_default_pricing
from cost_model.analytical_model import CostModel
from cost_model.crossover import CrossoverSolver
from cost_model.simulator import CostSimulator

# Inline plotting
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print('Imports OK')

## 1. Generate Simulated Measurement Data

If you haven't deployed to cloud, generate synthetic latency data
based on realistic distributions.

In [ ]:
from loadtest.sweep_runner import run_simulated_sweep, load_config

config = load_config(os.path.join(PROJECT_ROOT, 'loadtest', 'config.yaml'))
output_dir = os.path.join(PROJECT_ROOT, 'analysis', 'results', 'raw')

combined_path = run_simulated_sweep(config, output_dir=output_dir)
print(f'\nData generated: {combined_path}')

## 2. Cost Model Simulation

Run the analytical cost model across all request rates and compute
cost per 1000 inferences for each deployment mode.

In [ ]:
simulator = CostSimulator()
results_dir = os.path.join(PROJECT_ROOT, 'analysis', 'results')

# Run full simulation
comparison_df = simulator.simulate_full_sweep(output_dir=results_dir)
comparison_df

## 3. Crossover Analysis

Find the exact request rates where the cheapest option changes.

In [ ]:
solver = CrossoverSolver()
crossovers = solver.find_all_crossovers()

# Analytical crossover (no cold-start effects)
analytical = solver.analytical_crossover_serverless_container()
print(f'Analytical crossover (Serverless <-> Container): {analytical:.3f} req/s ({analytical*60:.1f} req/min)')

# Numerical crossover (with cold-start modelling)
numerical = solver.numerical_crossover('serverless', 'container')
print(f'Numerical crossover (with cold starts): {numerical:.3f} req/s ({numerical*60:.1f} req/min)')

# Serverless vs Managed
sm_xover = solver.numerical_crossover('serverless', 'managed')
if sm_xover:
    print(f'Serverless <-> Managed crossover: {sm_xover:.3f} req/s ({sm_xover*60:.1f} req/min)')

print('\nFull crossover details:')
for name, data in crossovers.items():
    print(f'  {name}: {data}')

## 4. Figure 1 — Cost Crossover Plot

The central figure showing cost per 1000 inferences vs request rate
with the crossover point annotated.

In [ ]:
from analysis.plot_helpers import (
    plot_cost_crossover, plot_latency_distribution,
    plot_cold_start_analysis, plot_sensitivity, plot_model_validation,
    COLORS, MODE_LABELS
)

sweep_df = pd.read_csv(os.path.join(results_dir, 'cost_sweep_smooth.csv'))

# Load measured analysis if available
analysis_path = os.path.join(results_dir, 'measured_analysis.csv')
analysis_df = pd.read_csv(analysis_path) if os.path.exists(analysis_path) else None

fig = plot_cost_crossover(sweep_df, crossovers, analysis_df)
plt.show()

## 5. Analyze Measured Data

Compare measured latencies against the cost model predictions.

In [ ]:
measured_path = os.path.join(results_dir, 'raw', 'combined_results.csv')

if os.path.exists(measured_path):
    analysis_df = simulator.analyze_measured_data(measured_path, results_dir)
    display(analysis_df)
else:
    print('No measured data found. Run step 1 first.')

## 6. Figure 2 — Latency Distribution

In [ ]:
if os.path.exists(measured_path):
    measured_df = pd.read_csv(measured_path)
    fig = plot_latency_distribution(measured_df)
    plt.show()

## 7. Figure 3 — Cold-Start Analysis

In [ ]:
if os.path.exists(measured_path):
    fig = plot_cold_start_analysis(measured_df)
    plt.show()

## 8. Sensitivity Analysis

How does the crossover point shift when we vary:
- Inference duration (model complexity)
- Lambda memory allocation
- EC2 hourly cost

In [ ]:
# Generate sensitivity data
sensitivity_results = simulator.generate_sensitivity_report(results_dir)

# Plot
fig = plot_sensitivity(sensitivity_results)
plt.show()

## 9. Figure 5 — Model Validation

In [ ]:
if analysis_df is not None and len(analysis_df) > 0:
    fig = plot_model_validation(analysis_df)
    plt.show()
else:
    print('Run measured data analysis first (step 5).')

## 10. Summary Table

Recommendation table for practitioners.

In [ ]:
model = CostModel()

summary_data = []
test_rates = [0.017, 0.1, 0.5, 1.0, 5.0, 10.0, 30.0, 100.0]

for rate in test_rates:
    cheapest, cost = model.find_cheapest(rate)
    summary_data.append({
        'Rate (req/s)': rate,
        'Rate (req/min)': rate * 60,
        'Serverless ($/1k)': f'${model.serverless_cost(rate):.4f}',
        'Container ($/1k)': f'${model.container_cost(rate):.4f}',
        'Managed ($/1k)': f'${model.managed_cost(rate):.4f}',
        'Cheapest': cheapest,
        'Monthly (cheapest)': f'${model.serverless_monthly(rate) if cheapest=="serverless" else model.container_monthly(rate):.2f}',
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

print(f'\n--- Crossover Point ---')
print(f'Serverless becomes more expensive than Container at: {numerical:.2f} req/s ({numerical*60:.0f} req/min)')
print(f'\nBelow {numerical:.1f} req/s: Use SERVERLESS')
print(f'Above {numerical:.1f} req/s: Use CONTAINER')
print(f'Managed endpoint is always more expensive than container (same model, higher hourly rate).')

## 11. Parameterised Formula for Practitioners

Apply this formula with your own pricing and model characteristics.

In [ ]:
print('=' * 65)
print('  CROSSOVER FORMULA')
print('=' * 65)
print()
print('  lambda_star = C_hour / ((C_req + C_gb_sec * M * D) * 3600)')
print()
print('  Where:')
print('    lambda_star = crossover request rate (req/s)')
print('    C_hour      = container hourly cost ($)')
print('    C_req       = serverless per-request cost ($)')
print('    C_gb_sec    = serverless per-GB-second cost ($)')
print('    M           = serverless memory (GB)')
print('    D           = average inference duration (seconds)')
print()
print('  Example with default AWS pricing:')
p = get_default_pricing()
C_hour = p.container.cost_per_hour
C_req = p.serverless.cost_per_request
C_gb = p.serverless.cost_per_gb_second
M = p.serverless.memory_gb
D = p.serverless.avg_duration_sec
lam = C_hour / ((C_req + C_gb * M * D) * 3600)
print(f'    C_hour   = ${C_hour}')
print(f'    C_req    = ${C_req}')
print(f'    C_gb_sec = ${C_gb}')
print(f'    M        = {M} GB')
print(f'    D        = {D} sec ({D*1000:.0f} ms)')
print(f'    lambda*  = {lam:.3f} req/s = {lam*60:.1f} req/min')
print()
print('=' * 65)